# ✍️ Introduction to Generation in RAG

## Where Are We?

```
1. Documents → Embeddings   ✅
2. Query → Retrieval        ✅
3. Query → Better Search    ✅
4. Context → Generation     ← You are here
```

Retrieval gives us **relevant documents**.  
Generation takes those documents and produces a **final answer**.

---

## The Generation Step

```
User Question
     +
Retrieved Documents
     ↓
  [Prompt]
     ↓
   LLM
     ↓
  Answer
```

The LLM reads the retrieved documents as **context** and generates an answer grounded in them.

This is what makes RAG different from plain LLM usage:
- **Plain LLM**: answers from training data alone → can hallucinate
- **RAG**: answers grounded in retrieved facts → more reliable

## What This Section Covers

| Notebook | Topic |
|---|---|
| `01_Prompt_Engineering.ipynb` | How to structure the prompt for RAG |
| `02_Context_Window_Management.ipynb` | What to do when context is too long |
| `03_Grounding_and_Citations.ipynb` | Make the LLM cite its sources |
| `04_Handling_Edge_Cases.ipynb` | When no docs are found, conflicting info, etc. |

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


In [ ]:
# !pip install anthropic  # For real LLM calls (optional)
# All notebooks in this section simulate LLM responses so they run without an API key

## 1. The Simplest Possible RAG Generation

In [ ]:
# Simulated retrieval output — 3 documents retrieved
retrieved_docs = [
    "Dropout randomly deactivates neurons during training to prevent overfitting.",
    "Dropout is applied only during training. At inference, all neurons are active.",
    "A typical dropout rate is 0.2 to 0.5, meaning 20–50% of neurons are dropped per step.",
]

user_question = "What is dropout and how is it used?"

print("Retrieved docs:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  {i}. {doc}")

In [ ]:
# Build the prompt — the simplest RAG prompt pattern

context = "\n".join(retrieved_docs)  # Join all docs into one block

prompt = f"""Use the context below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {user_question}
Answer:"""

print(prompt)

In [ ]:
answer = llm(prompt, max_tokens=300)


print("LLM Answer:")
print("-" * 50)
print(answer)

## 2. What Can Go Wrong in Generation

In [ ]:
# Problem 1: Bad prompt → LLM ignores the context

bad_prompt = f"""Answer this question: {user_question}"""
# No context at all! LLM will answer from its training data → potential hallucination

print("❌ Bad prompt (no context):")
print(bad_prompt)

print("\n✅ Good prompt (with context and clear instruction):")
print("   'Use the context below to answer...' + context + question")

In [ ]:
# Problem 2: Too many docs → context window overflow

# Most LLMs have a context window limit (e.g. 128k tokens for Claude)
# If retrieved docs are too long, the prompt gets truncated

# Quick token estimator (rough: 1 token ≈ 4 characters)
def estimate_tokens(text):
    return len(text) // 4

context_tokens = estimate_tokens(context)
question_tokens = estimate_tokens(user_question)
total_tokens = context_tokens + question_tokens + 50  # 50 for prompt template

print(f"Context tokens  (approx): {context_tokens}")
print(f"Question tokens (approx): {question_tokens}")
print(f"Total tokens    (approx): {total_tokens}")
print(f"\n✅ Fine for this example — but with 50 long docs this would overflow!")
print("   See 02_Context_Window_Management.ipynb for how to handle this.")

In [ ]:
# Problem 3: No relevant docs retrieved → LLM should say "I don't know"

irrelevant_docs = [
    "The Eiffel Tower is 330 meters tall.",
    "Paris is the capital of France.",
]

context_irrelevant = "\n".join(irrelevant_docs)

prompt_with_fallback = f"""Use the context below to answer the question.
If the answer is not in the context, say exactly: "I don't have enough information to answer this."

Context:
{context_irrelevant}

Question: {user_question}
Answer:"""

print("Prompt with irrelevant context + explicit fallback instruction:")
print(prompt_with_fallback)
print("\n→ LLM should say: 'I don't have enough information...'")
print("   See 04_Handling_Edge_Cases.ipynb for more on this.")

## 3. The RAG Generation Checklist

Before you send a prompt to the LLM:

```
✅ Is the context included in the prompt?
✅ Does the prompt tell the LLM to use the context?
✅ Is there a fallback for when context doesn't have the answer?
✅ Are the docs short enough to fit in the context window?
✅ Is the most relevant doc first? (LLMs pay more attention to the beginning)
```

---
Next: `01_Prompt_Engineering.ipynb` — learn the best prompt templates for RAG.